# HDB Price Prediction Regression Model

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score,
    classification_report,
)

## 1. Analyse Dataset

In [ ]:
# Load dataset
# df = pd.read_csv('HDB_Resale_Prices.csv')
# df = pd.read_csv('hdb_prices_2017.csv')

# Display first few rows
df.head(5)

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0


In [10]:
# Get random samples
df.sample(5)

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
3529,2025-08,BUKIT BATOK,5 ROOM,626,BT BATOK CTRL,07 TO 09,119.0,Improved,1997,70 years 11 months,850000.0
16085,2025-03,SEMBAWANG,5 ROOM,101A,CANBERRA ST,10 TO 12,113.0,Improved,2020,94 years 11 months,808888.0
15014,2025-01,QUEENSTOWN,4 ROOM,82,STRATHMORE AVE,22 TO 24,108.0,Model A,1997,71 years 10 months,990000.0
24317,2025-04,YISHUN,4 ROOM,327,YISHUN RING RD,01 TO 03,84.0,Simplified,1988,62 years 03 months,465000.0
11340,2025-01,JURONG WEST,EXECUTIVE,669B,JURONG WEST ST 64,10 TO 12,130.0,Apartment,2000,74 years 07 months,705000.0


In [4]:
# Dataset shape (rows, columns)
print(f'(rows, columns): {df.shape}')

# Column names
print(f'Columns: {df.columns.tolist()}')

print('\n--- DATASET INFO ---')
df.info()

print('\n--- DATASET DESCRIPTION ---')
df.describe()

(rows, columns): (227425, 11)
Columns: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price']

--- DATASET INFO ---
<class 'pandas.DataFrame'>
RangeIndex: 227425 entries, 0 to 227424
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   month                227425 non-null  str    
 1   town                 227425 non-null  str    
 2   flat_type            227425 non-null  str    
 3   block                227425 non-null  str    
 4   street_name          227425 non-null  str    
 5   storey_range         227425 non-null  str    
 6   floor_area_sqm       227425 non-null  float64
 7   flat_model           227425 non-null  str    
 8   lease_commence_date  227425 non-null  int64  
 9   remaining_lease      227425 non-null  str    
 10  resale_price         227425 non-null  float64
dtypes: float64

,floor_area_sqm,lease_commence_date,resale_price
count,227425.000000,227425.000000,2.274250e+05
mean,96.731798,1996.484173,5.275790e+05
std,24.017348,14.325668,1.882345e+05
min,31.000000,1966.000000,1.400000e+05
25%,81.000000,1985.000000,3.880000e+05
50%,93.000000,1997.000000,4.970000e+05
75%,112.000000,2012.000000,6.330000e+05
max,366.700000,2021.000000,1.700000e+06


In [12]:
# Check for null values
print("\n--- NULL VALUES CHECK ---")
df.isnull()

# Count null values per column
df.isnull().sum()


--- NULL VALUES CHECK ---


month                  0
town                   0
flat_type              0
block                  0
street_name            0
storey_range           0
floor_area_sqm         0
flat_model             0
lease_commence_date    0
remaining_lease        0
resale_price           0
dtype: int64

In [13]:


# Percentage of missing values
print('\n--- PERCENTAGE OF MISSING VALUES ---')
print((df.isnull().sum() / len(df)) * 100)



--- PERCENTAGE OF MISSING VALUES ---
month                  0.0
town                   0.0
flat_type              0.0
block                  0.0
street_name            0.0
storey_range           0.0
floor_area_sqm         0.0
flat_model             0.0
lease_commence_date    0.0
remaining_lease        0.0
resale_price           0.0
dtype: float64


Convert ```remaining_lease_years``` into float

In [14]:
# Extract years and months separately
df["years"] = df["remaining_lease"].str.extract(r"(\d+) years").astype(float).fillna(0)
df["months"] = df["remaining_lease"].str.extract(r"(\d+) month").astype(float).fillna(0)

# Convert to decimal years
df["remaining_lease_float"] = df["years"] + (df["months"] / 12)

# Check the new columns
df[["remaining_lease", "years", "months", "remaining_lease_float"]].head(10)

,remaining_lease,years,months,remaining_lease_float
0,53 years 01 month,53.0,1.0,53.083333
1,52 years 10 months,52.0,10.0,52.833333
2,52 years 02 months,52.0,2.0,52.166667
3,51 years 11 months,51.0,11.0,51.916667
4,86 years,86.0,0.0,86.000000
5,51 years 04 months,51.0,4.0,51.333333
6,51 years 02 months,51.0,2.0,51.166667
7,51 years 05 months,51.0,5.0,51.416667
8,50 years 07 months,50.0,7.0,50.583333
9,60 years 02 months,60.0,2.0,60.166667


In [15]:
print("\n--- DATASET INFO (After Processing) ---")
df.info()


--- DATASET INFO (After Processing) ---
<class 'pandas.DataFrame'>
RangeIndex: 25094 entries, 0 to 25093
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   month                  25094 non-null  str    
 1   town                   25094 non-null  str    
 2   flat_type              25094 non-null  str    
 3   block                  25094 non-null  str    
 4   street_name            25094 non-null  str    
 5   storey_range           25094 non-null  str    
 6   floor_area_sqm         25094 non-null  float64
 7   flat_model             25094 non-null  str    
 8   lease_commence_date    25094 non-null  int64  
 9   remaining_lease        25094 non-null  str    
 10  resale_price           25094 non-null  float64
 11  years                  25094 non-null  float64
 12  months                 25094 non-null  float64
 13  remaining_lease_float  25094 non-null  float64
dtypes: float64(5), int64(1),

## Data Processing

1. Replace ```storey_range``` with midpoint of the range as a integer. This allows the model to learn relationships between the storeys (eg. higher floors = higher price)

In [16]:
df["floor"] = df["storey_range"].str.split(" TO ", expand=True).astype(float).mean(axis=1)

df[["storey_range", "floor"]].head(5)

,storey_range,floor
0,04 TO 06,5.0
1,07 TO 09,8.0
2,10 TO 12,11.0
3,10 TO 12,11.0
4,13 TO 15,14.0


In [17]:
y = df['resale_price']

feature_columns = ['town', 'flat_type', 'floor', 'floor_area_sqm', 'flat_model', 'remaining_lease_float']

X = df[feature_columns]

In [18]:
# Categorical features (need encoding)
categorical_features = ["town", "flat_type", "flat_model"]

In [19]:
X_encoded = pd.get_dummies(X, columns=categorical_features, drop_first=True)

print(f"Original shape: {X.shape}")
print(f"After encoding: {X_encoded.shape}")

Original shape: (25094, 6)
After encoding: (25094, 54)


## Train-test-validation split

In [20]:
# First split: separate test set (15%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X_encoded, y, test_size=0.15, random_state=42
)

# Second split: separate validation set from remaining data (15% of total = ~18% of temp)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42  # 0.15/0.85 ≈ 0.176
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

Training set: 17575 samples
Validation set: 3754 samples
Test set: 3765 samples


For CatBoost, we don't need one-hot encoding. Generate dataset splits for CatBoost

In [21]:
X_raw = df[feature_columns].copy()
X_raw_temp, X_raw_test, y_temp2, y_test2 = train_test_split(
    X_raw, y, test_size=0.15, random_state=42
)
X_raw_train, X_raw_val, y_raw_train, y_raw_val = train_test_split(
    X_raw_temp, y_temp2, test_size=0.176, random_state=42
)
cat_feature_indices = [X_raw.columns.get_loc(c) for c in categorical_features]

## Different Models and Results

### Evaluation Function

In [ ]:
def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    r2   = r2_score(y_true, y_pred)
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  RMSE:   ${rmse:>12,.2f}")
    print(f"  MAE:    ${mae:>12,.2f}")
    print(f"  MAPE:    {mape:>11.4f}%")
    print(f"  R²:      {r2:>11.4f}")
    return {"model": name, "rmse": rmse, "mae": mae, "mape": mape, "r2": r2}

results = []

How to interpret R²:
| Performance Level | R² Score    | Expected for HDB           |
| ----------------- | ----------- | -------------------------- |
| Poor              | < 0.70      | Missing key features       |
| Fair              | 0.70 - 0.80 | Basic model                |
| Good              | 0.80 - 0.90 | Solid performance          |
| Excellent         | 0.90 - 0.95 | Very Strong performance        |
| Exceptional       | > 0.95      | Rare, possible overfitting |

### Random Forest

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1,
)

rf_model.fit(X_train, y_train)
results.append(evaluate("Random Forest (baseline)", y_test, rf_model.predict(X_test)))


  Random Forest (baseline)
  RMSE:   $   48,247.13
  MAE:    $   32,828.20
  MAPE:         4.9023%
  R²:           0.9437


### XGBoost

In [24]:
import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,            # L1 regularisation
    reg_lambda=1.0,           # L2 regularisation
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50, # stop if val loss doesn't improve for 50 rounds
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)
print(f"\n  XGBoost best iteration: {xgb_model.best_iteration}")
results.append(evaluate("XGBoost", y_test, xgb_model.predict(X_test)))


  XGBoost best iteration: 748

  XGBoost
  RMSE:   $   42,123.39
  MAE:    $   28,419.95
  MAPE:         4.2713%
  R²:           0.9571


### LightGBM

In [25]:
import lightgbm as lgb

lgb_model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
)

lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(50),   # stop if val loss doesn't improve for 50 rounds
        lgb.log_evaluation(0),    # suppress per-round logging
    ],
)
print(f"\n  LightGBM best iteration: {lgb_model.best_iteration_}")
results.append(evaluate("LightGBM", y_test, lgb_model.predict(X_test)))

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000542 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 485
[LightGBM] [Info] Number of data points in the train set: 17575, number of used features: 46
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Start training from score 653231.085398
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

### CatBoost

In [26]:
from catboost import CatBoostRegressor

cb_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3,
    subsample=0.8,
    random_seed=42,
    early_stopping_rounds=50,
    verbose=0,                # silent training
)

cb_model.fit(
    X_raw_train, y_raw_train,
    eval_set=(X_raw_val, y_raw_val),
    cat_features=cat_feature_indices,
)
print(f"\n  CatBoost best iteration: {cb_model.get_best_iteration()}")
results.append(evaluate("CatBoost", y_test2, cb_model.predict(X_raw_test)))


  CatBoost best iteration: 999

  CatBoost
  RMSE:   $   41,743.56
  MAE:    $   29,003.04
  MAPE:         4.3774%
  R²:           0.9578


### Summary of results

In [27]:
print("\n")
print("=" * 70)
print("  MODEL COMPARISON SUMMARY  (test set)")
print("=" * 70)
print(f"  {'Model':<28} {'RMSE':>12} {'MAE':>12} {'MAPE %':>10} {'R²':>8}")
print("-" * 70)
for r in results:
    print(
        f"  {r['model']:<28} "
        f"${r['rmse']:>11,.0f} "
        f"${r['mae']:>11,.0f} "
        f"{r['mape']:>9.2f}% "
        f"{r['r2']:>7.4f}"
    )
print("=" * 70)



  MODEL COMPARISON SUMMARY  (test set)
  Model                                RMSE          MAE     MAPE %       R²
----------------------------------------------------------------------
  Random Forest (baseline)     $     48,247 $     32,828      4.90%  0.9437
  XGBoost                      $     42,123 $     28,420      4.27%  0.9571
  LightGBM                     $     42,803 $     29,348      4.42%  0.9557
  CatBoost                     $     41,744 $     29,003      4.38%  0.9578


We can observe that CatBoost results in the lowest RMSE, beating RF by ~11k.

### Custom Input

In [28]:
sample_input = {
    "town": "WOODLANDS",
    "flat_type": "5 ROOM",
    "storey_range": "04 TO 06",
    "floor_area_sqm": 112.0,
    "flat_model": "Improved",
    "remaining_lease_years": 86.8333333333,
}

input_df = pd.DataFrame([sample_input])

input_encoded = pd.get_dummies(input_df, columns=categorical_features, drop_first=True)

# align to training feature columns (missing dummies become 0; extra cols dropped)
input_encoded = input_encoded.reindex(columns=X_encoded.columns, fill_value=0)

# Make prediction
predicted_price = rf_model.predict(input_encoded)[0]

print(f"Predicted resale price: ${predicted_price:,.2f}")

Predicted resale price: $697,640.15


NameError: name 'y_pred' is not defined